# Manual labeling — Jupyter / VSCode

Browse the regenerated 6-panel sample plots one by one and record your manual
label. After every click the result is appended to `manual_annotations.csv`
in the same sample folder, so you can close the notebook and pick up where
you left off.

1. Run **cell 1** to load helpers.
2. Run **cell 2**, pick the sample folder you want to label
   (`tune` / `val_random` / `val_stratified`).
3. Run **cell 3** to launch the labeling UI for that folder. Click *Plume / No Plume / Unsure / Skip*.
4. Re-run cell 3 with a different folder later to label another set.
5. Run **cell 4** for the agreement / confusion matrix summary.

In [1]:
# Cell 1 — imports & helpers
import os
import datetime as dt
from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

SAMPLES_ROOT = Path('/net/fs06/d3/rzhuang/TROPOMI/data/world/paper_figures/labeling_samples')
ANNOTATIONS_FILE = 'manual_annotations.csv'

LABEL_OPTIONS = [
    ('plume',     '\u2713 Plume',     'success'),
    ('no_plume',  '\u2717 No Plume',  'danger'),
    ('unsure',    '? Unsure',         'warning'),
    ('skip',      '\u23ed Skip',      ''),
]

def list_sample_folders():
    return [p for p in sorted(SAMPLES_ROOT.iterdir())
            if p.is_dir() and (p / 'sampled_emission_snapshots.csv').exists()]

print('available folders:')
for f in list_sample_folders():
    n = sum(1 for _ in f.glob('sampled_location_*.png'))
    print(f'  - {f.name:18s}  ({n} PNGs)')

available folders:
  - tune                (200 PNGs)
  - val_random          (400 PNGs)
  - val_stratified      (400 PNGs)


In [2]:
# Cell 2 — pick a folder. Re-run to switch sets.
SAMPLES_DIR = SAMPLES_ROOT / 'val_random'   # <-- edit me: tune | val_random | val_stratified
assert SAMPLES_DIR.exists() and (SAMPLES_DIR / 'sampled_emission_snapshots.csv').exists(), \
    f'No sampled_emission_snapshots.csv under {SAMPLES_DIR}'
print(f'Selected: {SAMPLES_DIR}')

Selected: /net/fs06/d3/rzhuang/TROPOMI/data/world/paper_figures/labeling_samples/val_random


In [3]:
# Cell 3 — labeling UI with built-in filter (all / FP / FN / disagreement / ...)
class LabelReviewer:
    """Single labeling UI; the dropdown at the top filters which subset of the
    folder is shown. Filters that depend on existing annotations only show
    samples already labeled."""

    FILTER_OPTIONS = [
        ('All',                    'all'),
        ('Disagreements (FP+FN)',  'disagreement'),
        ('False positives (algo=plume, you=no_plume)', 'fp'),
        ('False negatives (algo=no_plume, you=plume)', 'fn'),
        ('Unsure',                 'unsure'),
        ('Algo says plume',        'algo_plume'),
        ('Algo says no plume',     'algo_no_plume'),
    ]

    def __init__(self, samples_dir: Path):
        self.samples_dir = samples_dir
        self.csv_path = samples_dir / 'sampled_emission_snapshots.csv'
        self.ann_path = samples_dir / ANNOTATIONS_FILE

        self.full_df     = self._load_samples()
        self.annotations = self._load_annotations()
        self.df          = self.full_df            # currently shown subset
        self.filter_mode = 'all'
        self.idx         = self._first_unlabeled_idx()

        self._build_ui()
        self._refresh()

    # ---- data ----
    def _load_samples(self):
        df = pd.read_csv(self.csv_path)
        def png_for(row):
            loc = row.get('location', 'UnknownLocation')
            iso = row.get('country', row.get('ISO3', 'UnknownISO'))
            t   = row.get('utc_time', 'UnknownTime')
            return self.samples_dir / f'sampled_location_{loc}_{iso}_{t}.png'
        df['_png'] = df.apply(png_for, axis=1)
        df['_png_exists'] = df['_png'].apply(lambda p: p.exists())
        df['_sample_id']  = df['_png'].apply(lambda p: p.stem)
        missing = (~df['_png_exists']).sum()
        if missing:
            print(f'Warning: {missing} of {len(df)} CSV rows have no PNG on disk.')
        df = df[df['_png_exists']].reset_index(drop=True)
        if df.empty:
            raise RuntimeError(f'No samples to show for {self.samples_dir}')
        return df

    def _load_annotations(self):
        if not self.ann_path.exists():
            return {}
        a = pd.read_csv(self.ann_path)
        return {r['sample_id']: dict(r) for _, r in a.iterrows()}

    def _save_annotation(self, sample_id, label, row):
        self.annotations[sample_id] = {
            'sample_id':   sample_id,
            'location':    row.get('location'),
            'country':     row.get('country', row.get('ISO3', '')),
            'utc_time':    row.get('utc_time'),
            'algo_label':  bool(row.get('plume_label', False)),
            'human_label': label,
            'timestamp':   dt.datetime.now().isoformat(timespec='seconds'),
        }
        out = pd.DataFrame(list(self.annotations.values()))
        out.to_csv(self.ann_path, index=False)

    def _first_unlabeled_idx(self):
        # When a filter is applied, start at row 0 so the user sees every
        # match in order rather than auto-skipping already-labelled rows.
        if self.filter_mode != 'all':
            return 0
        for i, row in self.df.iterrows():
            sid = row['_sample_id']
            if sid not in self.annotations or not self.annotations[sid].get('human_label'):
                return i
        return len(self.df) - 1

    # ---- filter ----
    def _apply_filter(self, mode):
        self.filter_mode = mode
        if mode == 'all':
            ids = None
        else:
            if not self.annotations:
                print(f"Filter {mode!r} needs existing annotations; none yet.")
                ids = []
            else:
                ann = pd.DataFrame(list(self.annotations.values()))
                ann['algo_label'] = ann['algo_label'].astype(bool)
                if mode == 'fp':
                    sel = ann[(ann.algo_label) & (ann.human_label == 'no_plume')]
                elif mode == 'fn':
                    sel = ann[(~ann.algo_label) & (ann.human_label == 'plume')]
                elif mode == 'disagreement':
                    sel = ann[((ann.algo_label) & (ann.human_label == 'no_plume')) |
                              ((~ann.algo_label) & (ann.human_label == 'plume'))]
                elif mode == 'unsure':
                    sel = ann[ann.human_label == 'unsure']
                elif mode == 'algo_plume':
                    sel = ann[ann.algo_label]
                elif mode == 'algo_no_plume':
                    sel = ann[~ann.algo_label]
                else:
                    sel = ann
                ids = set(sel['sample_id'].tolist())

        if ids is None:
            self.df = self.full_df
        else:
            self.df = self.full_df[self.full_df['_sample_id'].isin(ids)].reset_index(drop=True)

        if len(self.df) == 0:
            print(f"No samples match filter {mode!r}.")
            self.df = self.full_df.iloc[:0].copy()  # keep schema
        self.idx = self._first_unlabeled_idx() if len(self.df) else 0
        self._refresh()

    # ---- ui ----
    def _build_ui(self):
        self.title  = widgets.HTML()
        self.image  = widgets.Image(format='png', width=900)
        self.meta   = widgets.HTML()
        self.algo   = widgets.HTML()
        self.stats  = widgets.HTML()

        # Filter dropdown
        self.filter_dd = widgets.Dropdown(
            options=self.FILTER_OPTIONS, value='all',
            description='Show:',
            layout=widgets.Layout(width='420px'))
        self.filter_dd.observe(
            lambda c: self._apply_filter(c['new']) if c['name'] == 'value' else None)

        # Label buttons
        label_btns = []
        for value, text, style in LABEL_OPTIONS:
            b = widgets.Button(description=text, button_style=style,
                                layout=widgets.Layout(width='150px', height='50px'))
            b.on_click(lambda _b, v=value: self._record(v))
            label_btns.append(b)
        labels_box = widgets.HBox(label_btns)

        # Navigation
        prev_btn   = widgets.Button(description='← Prev')
        next_btn   = widgets.Button(description='Next →')
        resume_btn = widgets.Button(description='Resume ⏭', button_style='info')
        prev_btn.on_click(lambda _: self._prev())
        next_btn.on_click(lambda _: self._next())
        resume_btn.on_click(lambda _: self._resume())
        nav_box = widgets.HBox([prev_btn, next_btn, resume_btn])

        right = widgets.VBox([self.filter_dd, self.meta, self.algo,
                              labels_box, nav_box, self.stats])
        self.layout = widgets.VBox([self.title,
                                    widgets.HBox([self.image, right])])
        display(self.layout)

    # ---- nav ----
    def _record(self, label):
        if len(self.df) == 0:
            return
        row = self.df.iloc[self.idx]
        self._save_annotation(row['_sample_id'], label, row)
        # Advance, but if we're inside a filtered view that depends on the
        # annotation we just rewrote (fp / fn / disagreement / unsure), the
        # row may no longer match -- recompute the filter so the list stays
        # consistent.
        if self.filter_mode in ('fp', 'fn', 'disagreement', 'unsure'):
            self._apply_filter(self.filter_mode)
        else:
            self._next()

    def _next(self):
        if len(self.df) == 0:
            return
        if self.idx < len(self.df) - 1:
            self.idx += 1
        self._refresh()

    def _prev(self):
        if len(self.df) == 0:
            return
        if self.idx > 0:
            self.idx -= 1
        self._refresh()

    def _resume(self):
        self.idx = self._first_unlabeled_idx() if len(self.df) else 0
        self._refresh()

    # ---- render ----
    def _refresh(self):
        if len(self.df) == 0:
            self.title.value = (f"<h3 style='margin:4px 0;color:#888'>"
                                f"{self.samples_dir.name} — no samples match "
                                f"filter '{self.filter_mode}'</h3>")
            self.image.value = b''
            self.meta.value = self.algo.value = self.stats.value = ''
            return

        row = self.df.iloc[self.idx]
        sid = row['_sample_id']
        ann = self.annotations.get(sid, {})

        try:
            self.image.value = row['_png'].read_bytes()
        except Exception as e:
            self.image.value = b''
            print(f'image load error: {e}')

        labelled = sum(1 for v in self.annotations.values() if v.get('human_label'))
        you = ann.get('human_label', '— unlabeled —')
        filt = '' if self.filter_mode == 'all' else f' &middot; filter: <code>{self.filter_mode}</code>'
        self.title.value = (
            f"<h3 style='margin:4px 0'>{self.samples_dir.name}{filt} — "
            f"sample <b>{self.idx + 1}</b> / {len(self.df)} "
            f"<span style='color:#888;font-weight:normal;font-size:13px'>"
            f"(folder labeled: {labelled} • your label here: <b>{you}</b>)</span></h3>")

        emis = None
        if 'NOx Mass (lbs)' in row and pd.notna(row['NOx Mass (lbs)']):
            emis = f"{row['NOx Mass (lbs)'] * 0.453592:.1f} kg/h"
        elif 'annual_nox_emission' in row and pd.notna(row['annual_nox_emission']):
            emis = f"{row['annual_nox_emission']:.1f}"

        latlon = '?'
        if pd.notna(row.get('latitude')) and pd.notna(row.get('longitude')):
            latlon = f"{row['latitude']:.3f}, {row['longitude']:.3f}"

        self.meta.value = (
            "<table style='font-family:monospace;font-size:13px'>"
            f"<tr><td><b>location</b></td><td>{row.get('location','?')}</td></tr>"
            f"<tr><td><b>country</b></td><td>{row.get('country', row.get('ISO3','?'))}</td></tr>"
            f"<tr><td><b>utc_time</b></td><td>{row.get('utc_time','?')}</td></tr>"
            f"<tr><td><b>lat,lon</b></td><td>{latlon}</td></tr>"
            f"<tr><td><b>emission</b></td><td>{emis or '?'}</td></tr>"
            f"<tr><td><b>continent</b></td><td>{row.get('continent','—')}</td></tr>"
            f"<tr><td><b>_dataset</b></td><td>{row.get('_dataset','—')}</td></tr>"
            "</table>")

        algo = bool(row.get('plume_label', False))
        self.algo.value = (
            f"<div style='margin:8px 0;padding:8px 12px;border-radius:4px;"
            f"background:{'#d4edda' if algo else '#f8d7da'};"
            f"color:{'#155724' if algo else '#721c24'};"
            f"font-weight:bold;font-size:14px;text-align:center'>"
            f"Algorithm: {'PLUME' if algo else 'NO PLUME'}</div>")

        labeled = [v for v in self.annotations.values()
                   if v.get('human_label') in ('plume', 'no_plume')]
        if labeled:
            tp = sum(1 for v in labeled if v['algo_label'] and v['human_label'] == 'plume')
            tn = sum(1 for v in labeled if not v['algo_label'] and v['human_label'] == 'no_plume')
            fp = sum(1 for v in labeled if v['algo_label'] and v['human_label'] == 'no_plume')
            fn = sum(1 for v in labeled if not v['algo_label'] and v['human_label'] == 'plume')
            n  = tp + tn + fp + fn
            agree = (tp + tn) / n * 100 if n else 0
            self.stats.value = (
                "<table style='font-family:monospace;font-size:13px;margin-top:8px'>"
                f"<tr><td colspan=3 style='padding-top:6px'><b>Agreement so far ({n} labeled): {agree:.1f}%</b></td></tr>"
                "<tr><td></td><td><b>You: plume</b></td><td><b>You: no plume</b></td></tr>"
                f"<tr><td><b>Algo: plume</b></td><td>{tp}</td><td>{fp}</td></tr>"
                f"<tr><td><b>Algo: no plume</b></td><td>{fn}</td><td>{tn}</td></tr>"
                "</table>")
        else:
            self.stats.value = "<i>label some samples to see agreement</i>"

reviewer = LabelReviewer(SAMPLES_DIR)


In [5]:
# Cell 4 — summary across all sample folders
from pathlib import Path
rows = []
for folder in list_sample_folders():
    a = folder / ANNOTATIONS_FILE
    if not a.exists():
        continue
    df = pd.read_csv(a)
    df = df[df['human_label'].isin(['plume', 'no_plume'])].copy()
    if df.empty:
        continue
    df['algo_label'] = df['algo_label'].astype(bool)
    tp = ((df.algo_label) & (df.human_label == 'plume')).sum()
    tn = ((~df.algo_label) & (df.human_label == 'no_plume')).sum()
    fp = ((df.algo_label) & (df.human_label == 'no_plume')).sum()
    fn = ((~df.algo_label) & (df.human_label == 'plume')).sum()
    rows.append(dict(folder=folder.name, n=len(df),
                     accuracy=f'{(tp+tn)/len(df)*100:.1f}%',
                     precision=f'{tp/(tp+fp)*100:.1f}%' if tp+fp else '-',
                     recall=f'{tp/(tp+fn)*100:.1f}%' if tp+fn else '-',
                     TP=tp, FP=fp, FN=fn, TN=tn))
if rows:
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print('no manual_annotations.csv found in any sample folder yet')

        folder   n accuracy precision recall  TP  FP  FN  TN
    val_random 400    98.0%     93.5%  93.5%  58   4   4 334
val_stratified 400    98.0%     88.0%  95.7%  44   6   2 348
